In [27]:
%load_ext autoreload
%autoreload 2
import os, sys
os.environ["OMP_NUM_THREADS"] = "2"

import matplotlib.pyplot                            as plt
import numpy                                        as np
import time
from scipy.optimize     import approx_fprime
from scipy              import linalg               as LA


from sop_lake.SOP                import SOP
from sop_lake.dmft_config        import load_sim_config
from sop_lake.data_io            import read_conv_history, read_dmft_data, read_vemb_data
from sop_lake.AIMSOP_utils       import reversed_AIMSOP
from sop_lake.embedding_utils    import frequency_axis, get_SigmaA_SOP, get_Gloc_SOP, get_new_vemb_SOP


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Color palettes
palette          = ["#822D8A","#D8BFD8"]
vik_red_palette  = ["#044F88","#4F93B5", "#902B05","#C26F40","#F4B400", "#1B9E77","#8BC34A", "#6A3D9A","#CE5B9E"]
palette_7        = ["#9C575E", "#8764A4", "#C67281", "#52A09C", "#5386C2", "#325277", "#E5877C", "#C4A277", "#627C4C"]
prx_palette      = ["#004181","#76BBFE","#F7899E", "#A90307"]                    # From paper Mario Motta
hc_palette       = ["#a50026","#d73027","#f46d43","#fdae61","#fee090","#ffffbf","#e0f3f8","#abd9e9","#74add1", "#4575b4","#313695"]
Ferretti_palette = ['#000000', '#c30201', '#ff6904', '#ffbd83', '#799a9d'] 


In [6]:
# Plots parameters
single_col_wd = 3.37  # in inches
double_col_wd = 7.0   # in inches
max_height    = 8.5   # in inches

plt.rcParams.update({
    "font.family": "DejaVu Sans",           # Also: "STIXGeneral"
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "lines.linewidth": 1.0,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.8,
    "xtick.major.size": 2.5,
    "xtick.labelsize": 10,
    "ytick.major.width": 0.8,
    "ytick.labelsize": 10,
    "axes.labelsize": 10,
    "ytick.major.size": 2.5,
})

def golden_height(wd):
    return (wd / 1.618)



# Testing `reversed_AIMSOP`,`get_SigmaA_SOP`, `get_Gloc_SOP`

In [ ]:
dir_name  = "/Users/acarbone/Desktop/tests/grid_free_tests/test_1"
sim_config = load_sim_config(dir_name + "/config.yaml")
w_list, Gimp_SOP, Gloc_list, SigmaA_list = read_dmft_data(dir_name + "/dmft_output.json")
conv_history = read_conv_history(dir_name + "/conv_output.json")

axis, eta_axis, num_pts, w_edges, matsubara_params = sim_config.embedding.axis, sim_config.embedding.eta_axis, sim_config.embedding.num_pts, sim_config.embedding.w_edges, sim_config.embedding.matsubara_params
w_list, w_sim_list = frequency_axis(axis, eta_axis, num_pts, matsubara_params, w_edges)
Gimp_list = Gimp_SOP.evaluate(w_sim_list)


In [46]:
ntot = Gimp_SOP.dim
const_term, B2_list, Omega_list = reversed_AIMSOP(Gimp_SOP)
B_SOP = SOP(B2_list,Omega_list)
rev_Gimp_list = - B_SOP.evaluate(w_sim_list) - const_term
rev_Gimp_list2 = [w * np.identity(ntot) - LA.inv(Gimp_list[iw]) for iw,w in enumerate(w_sim_list)]
print("Is reversed_AIMSOP correct? ", np.allclose(rev_Gimp_list, rev_Gimp_list2))


Is reversed_AIMSOP correct?  True


In [53]:
hA_1 = sim_config.system.get_single_particle_hA()       # Single-particle non-interacting Hamiltonian on the fragment A
mu   = conv_history["mu"][-1]
SOP_vemb = sim_config.get_input_variables()[3]
SigmaA_SOP, const_term_SigmaA = get_SigmaA_SOP(Gimp_SOP,SOP_vemb,mu,hA_1)


FileNotFoundError: [Errno 2] No such file or directory: 'config_input.yaml'

In [52]:
SigmaA_list2 = const_term_SigmaA + SigmaA_SOP.evaluate(w_sim_list)
print("Is get_SigmaA_SOP correct? ", np.allclose(SigmaA_list, SigmaA_list2))


Is get_SigmaA_SOP correct?  False


In [ ]:
ind = 10
print(SigmaA_list[ind],SigmaA_list2[ind])


[[0.47142727+0.00192314j 0.        +0.j        ]
 [0.        +0.j         0.47142727+0.00192314j]] [[4.43475296e-01+2.94910331e-02j 3.49395839e-15+3.58156655e-16j]
 [3.71627535e-15-6.37109943e-16j 4.43475296e-01+2.94910331e-02j]]
